# 🧹 Phase 1B + 1C — Cleaning & Feature Engineering
**Project:** Worker Productivity Dashboard | **Dataset:** Garment Employees Productivity

### What this notebook does
**1B — Cleaning**
1. Clip `actual_productivity` values above 1.0
2. Impute missing `wip` values with the column median
3. Cap outliers in skewed numerical columns
4. Drop columns not useful as model inputs (`date`, `team`)
5. One-hot encode categorical columns (`quarter`, `department`, `day`)

**1C — Feature Engineering**
6. Create `output_per_worker` — a productivity efficiency ratio
7. Create `overtime_flag` — binary indicator of any overtime worked
8. Audit final columns and verify zero nulls
9. Save `garment_clean.csv` and `feature_columns.pkl`

> ⚠️ **Note on scaling:** StandardScaler will be fit in Phase 2 *after* the train/test split.
> Fitting it here on the full dataset would leak test-set information into the model.

---
## Setup — Imports & Load Raw Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')

RAW_PATH     = '../data/raw/garments_worker_productivity.csv'
CLEANED_PATH = '../data/cleaned/garment_clean.csv'
MODEL_DIR    = '../model'

os.makedirs('../data/cleaned', exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

df = pd.read_csv(RAW_PATH)

print(f'✅ Raw data loaded: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'   Null count before cleaning: {df.isnull().sum().sum()}')

✅ Raw data loaded: 1197 rows × 15 columns
   Null count before cleaning: 506


---
## 1B.1 — Clip `actual_productivity` to Maximum 1.0

From EDA we know `max = 1.1204`. A productivity ratio cannot exceed 1.0 — these are measurement/data-entry artifacts. We clip rather than drop so we keep every row.

In [2]:
before_max = df['actual_productivity'].max()
n_above_1  = (df['actual_productivity'] > 1.0).sum()

df['actual_productivity'] = df['actual_productivity'].clip(upper=1.0)

after_max = df['actual_productivity'].max()

print(f'  Before clip — max: {before_max:.4f}   rows > 1.0: {n_above_1}')
print(f'  After  clip — max: {after_max:.4f}   rows > 1.0: {(df["actual_productivity"] > 1.0).sum()}')
print('✅ actual_productivity clipped to 1.0')

  Before clip — max: 1.1204   rows > 1.0: 37
  After  clip — max: 1.0000   rows > 1.0: 0
✅ actual_productivity clipped to 1.0


---
## 1B.2 — Impute Missing Values in `wip`

`wip` (Work In Progress) is the only column with nulls. We impute with the **median** rather than the mean because `wip` is right-skewed — the median is a more robust central estimate when outliers are present.

In [3]:
wip_null_before = df['wip'].isnull().sum()
wip_median      = df['wip'].median()

df['wip'] = df['wip'].fillna(wip_median)

wip_null_after = df['wip'].isnull().sum()

print(f'  wip median used for imputation : {wip_median:.2f}')
print(f'  Nulls before : {wip_null_before}')
print(f'  Nulls after  : {wip_null_after}')
print(f'  Total nulls remaining in dataset: {df.isnull().sum().sum()}')
print('✅ wip imputed')

  wip median used for imputation : 1039.00
  Nulls before : 506
  Nulls after  : 0
  Total nulls remaining in dataset: 0
✅ wip imputed


---
## 1B.3 — Cap Outliers in Skewed Numerical Columns

We use the **IQR fence method**: values beyond `Q3 + 1.5 × IQR` are capped at that fence. This trims extreme values without deleting rows.

Columns to check: `over_time`, `incentive`, `idle_time`, `idle_men`, `wip`

In [11]:
SKEWED_COLS = ['over_time', 'incentive', 'wip']

cap_log = []
for col in SKEWED_COLS:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    upper_fence = Q3 + 1.5 * IQR

    n_capped = (df[col] > upper_fence).sum()
    df[col]  = df[col].clip(upper=upper_fence)
    cap_log.append({'column': col, 'upper_fence': round(upper_fence, 2),
                    'rows_capped': n_capped})

cap_df = pd.DataFrame(cap_log)
print('Outlier Capping Summary:')
print(cap_df.to_string(index=False))
print('\n✅ Outliers capped')

Outlier Capping Summary:
   column  upper_fence  rows_capped
over_time      15240.0            0
incentive        125.0            0
      wip       1252.5            0

✅ Outliers capped


---
## 1B.4 — Drop Columns Not Useful as Model Inputs

- `date` — raw timestamp; the model doesn't need to know the calendar date
- `team` — team ID number; this is an arbitrary identifier, not a meaningful numeric feature

In [12]:
DROP_COLS = ['date', 'team']

df.drop(columns=DROP_COLS, inplace=True)

print(f'Dropped columns : {DROP_COLS}')
print(f'Shape after drop: {df.shape}')
print('✅ Columns dropped')

KeyError: "['date', 'team'] not found in axis"

---
## 1B.5 — One-Hot Encode Categorical Columns

Tree models like XGBoost can technically handle label-encoded categoricals, but one-hot encoding is safer and more transparent for this project — it avoids implying any ordinal relationship between categories.

Columns: `quarter`, `department`, `day`

> `drop_first=True` removes one dummy column per category to avoid the [dummy variable trap](https://en.wikipedia.org/wiki/Multicollinearity).

In [ ]:
CAT_COLS = ['quarter', 'department', 'day']

print('Unique values before encoding:')
for col in CAT_COLS:
    print(f'  {col}: {sorted(df[col].unique())}')

df = pd.get_dummies(df, columns=CAT_COLS, drop_first=True)

print(f'\nShape after encoding: {df.shape}')
print('\nNew columns added:')
new_cols = [c for c in df.columns if any(c.startswith(cat) for cat in CAT_COLS)]
print(new_cols)
print('\n✅ Categorical columns encoded')

---
## 1B.6 — Data Leakage Check

A leakage check confirms that `actual_productivity` (our target) cannot be mathematically derived from any input feature at inference time.

The one column to watch: `targeted_productivity`. It is the *quota set before the shift*, not calculated from the outcome — so it is safe to use as an input.

In [13]:
TARGET = 'actual_productivity'

# Verify target exists and is not in features
assert TARGET in df.columns, f'Target column missing!'
feature_cols = [c for c in df.columns if c != TARGET]

# Sanity: confirm actual_productivity is not perfectly correlated with any single feature
high_corr = df[feature_cols + [TARGET]].corr()[TARGET].drop(TARGET)
suspicious = high_corr[high_corr.abs() > 0.95]

if suspicious.empty:
    print('✅ No leakage detected — no feature has |correlation| > 0.95 with target')
else:
    print('⚠️  Potentially leaky features:')
    print(suspicious)

print(f'\n  Target column     : {TARGET}')
print(f'  Number of features: {len(feature_cols)}')

✅ No leakage detected — no feature has |correlation| > 0.95 with target

  Target column     : actual_productivity
  Number of features: 22


---
## 1C.1 — Feature Engineering

Two new features grounded in the EDA findings:

| New Feature | Formula | Why |
|-------------|---------|-----|
| `output_per_worker` | `targeted_productivity × no_of_workers` | Captures the *total expected output* of the team, not just the per-worker rate |
| `overtime_flag` | `1 if over_time > 0 else 0` | Separates 'any overtime' from 'how much' — the EDA showed many zero-overtime rows |

In [14]:
df['output_per_worker'] = df['targeted_productivity'] * df['no_of_workers']
df['overtime_flag']     = (df['over_time'] > 0).astype(int)

print('New features created:')
print(f"  output_per_worker — range: [{df['output_per_worker'].min():.2f}, "
      f"{df['output_per_worker'].max():.2f}]  mean: {df['output_per_worker'].mean():.2f}")
print(f"  overtime_flag     — value counts:")
print(df['overtime_flag'].value_counts().to_string())
print('\n✅ Features engineered')

New features created:
  output_per_worker — range: [1.40, 71.20]  mean: 25.07
  overtime_flag     — value counts:
overtime_flag
1    1166
0      31

✅ Features engineered


---
## 1C.2 — Final Column Audit

Before saving, confirm:
- Zero nulls remaining
- All columns are numeric (no lingering strings)
- `actual_productivity` max ≤ 1.0
- Shape looks right

In [15]:
bool_cols = df.select_dtypes(include='bool').columns.tolist()
df[bool_cols] = df[bool_cols].astype(int)

print('=' * 55)
print('FINAL DATASET AUDIT')
print('=' * 55)
print(f'  Shape                     : {df.shape}')
print(f'  Total nulls               : {df.isnull().sum().sum()}')
print(f'  actual_productivity max   : {df["actual_productivity"].max():.4f}')
print(f'  actual_productivity min   : {df["actual_productivity"].min():.4f}')
print(f'  Non-numeric columns       : {df.select_dtypes(exclude="number").columns.tolist()}')
print()
print('All columns and dtypes:')
print(df.dtypes.to_string())

FINAL DATASET AUDIT
  Shape                     : (1197, 23)
  Total nulls               : 0
  actual_productivity max   : 1.0000
  actual_productivity min   : 0.2337
  Non-numeric columns       : []

All columns and dtypes:
targeted_productivity    float64
smv                      float64
wip                      float64
over_time                  int64
incentive                  int64
idle_time                float64
idle_men                   int64
no_of_style_change         int64
no_of_workers            float64
actual_productivity      float64
quarter_Quarter2           int64
quarter_Quarter3           int64
quarter_Quarter4           int64
quarter_Quarter5           int64
department_finishing       int64
department_sweing          int64
day_Saturday               int64
day_Sunday                 int64
day_Thursday               int64
day_Tuesday                int64
day_Wednesday              int64
output_per_worker        float64
overtime_flag              int64


---
## 1C.3 — Save Outputs

Three files saved:
1. `data/cleaned/garment_clean.csv` — the full cleaned + engineered dataset
2. `model/feature_columns.pkl` — the exact list of input feature names the model expects
3. `model/target_column.pkl` — the target column name (for reference in Phase 2)

> The scaler (StandardScaler) is **not** saved here — it must be fit on training data only, which happens after the train/test split in Phase 2.

In [16]:
TARGET = 'actual_productivity'
feature_cols = [c for c in df.columns if c != TARGET]

# 1. Save cleaned CSV
df.to_csv(CLEANED_PATH, index=False)
print(f'✅ Saved: {CLEANED_PATH}')
print(f'   Shape: {df.shape}')

# 2. Save feature column list
joblib.dump(feature_cols, os.path.join(MODEL_DIR, 'feature_columns.pkl'))
print(f'\n✅ Saved: {MODEL_DIR}/feature_columns.pkl')
print(f'   Features ({len(feature_cols)}): {feature_cols}')

# 3. Save target column name
joblib.dump(TARGET, os.path.join(MODEL_DIR, 'target_column.pkl'))
print(f'\n✅ Saved: {MODEL_DIR}/target_column.pkl')
print(f'   Target: {TARGET}')

✅ Saved: ../data/cleaned/garment_clean.csv
   Shape: (1197, 23)

✅ Saved: ../model/feature_columns.pkl
   Features (22): ['targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers', 'quarter_Quarter2', 'quarter_Quarter3', 'quarter_Quarter4', 'quarter_Quarter5', 'department_finishing ', 'department_sweing', 'day_Saturday', 'day_Sunday', 'day_Thursday', 'day_Tuesday', 'day_Wednesday', 'output_per_worker', 'overtime_flag']

✅ Saved: ../model/target_column.pkl
   Target: actual_productivity


---
## ✅ Phase 1B + 1C Complete

| Step | Action | Status |
|------|--------|--------|
| 1B.1 | Clipped `actual_productivity` to max 1.0 | ✅ |
| 1B.2 | Imputed `wip` nulls with median | ✅ |
| 1B.3 | Capped outliers via IQR fence | ✅ |
| 1B.4 | Dropped `date` and `team` columns | ✅ |
| 1B.5 | One-hot encoded `quarter`, `department`, `day` | ✅ |
| 1B.6 | Confirmed no data leakage | ✅ |
| 1C.1 | Created `output_per_worker` and `overtime_flag` | ✅ |
| 1C.2 | Audited final shape and dtypes | ✅ |
| 1C.3 | Saved `garment_clean.csv` and `.pkl` files | ✅ |

**Next:** `notebooks/02_model_training.ipynb` → Phase 2